# Exploratory Data Analysis (EDA)
## ML Modeling Challenge - Wizeline

In [ ]:
import polars as pl

from src.eda_functions import (
    get_histograms,
    get_mutual_information,
    get_scatter_plots,
    get_spearman_matrix,
    get_train_test_comparison,
)


# 1. Carga de datos.

In [ ]:
# Datos de entrenamiento
df_train = pl.read_csv("data/training_data.csv")
print(f"Training data shape: {df_train.shape}")
df_train.head()

In [ ]:
# Datos de test
df_test = pl.read_csv("data/blind_test_data.csv")
print(f"Blind test data shape: {df_test.shape}")
df_test.head()

# 2. Pequeño vistazo a las características de las features.

In [ ]:
df_train.describe()

In [ ]:
df_test.describe()

NOTAS:
1. Se da un breve vistazo a como los estadísticos descritivos de cada feature.
2. NO se detecta presencia de valores pérdidos.

## 3. Análisis de la correlación no paramétrica de Spearman.

In [ ]:
get_spearman_matrix(df_train)

NOTAS:
1. Se usa correlación no parámetrica de Spearman porque no requiere supuesto de normalidad y es mas robusta ante outliers.
2. No hay evidencia de multicolinealidad entre las features.
3. No hay evidencia de relaciones lineales fuertes entre el Target y las features (ver última fila de la matriz). Tal vez una relación lineal discreta sea entre el Target y la feature 2.
4. Tal vez la relación entre el Target y las features es más compleja y, por ende, se necesitan modelos que capturen grados de complejidad mayores a la linealidad.

## 4. Cálculo de la Información Mutua (dependencia lineal y no lineal con target) y pre-selección de features.

In [ ]:
mi_df = get_mutual_information(df_train)

In [ ]:
list_mi_feature_selected = mi_df.filter(pl.col("mutual_info") > 0.01).sort("mutual_info", descending=True)['feature'].to_list()
list_mi_feature_selected

NOTAS:
1. La información mutua (MI) es una métrica que nos permite medir la dependencia (lineal o no lineal) entre dos variables, en este caso un Target y una feature. 
2. Mientras más alta sea el MI, más evidencia de dependencia.
3. La IM puede usarse para una selección de features antes de entrenamiento. 
4. La lista de variables "pre-seleccionadas" corresponde a las features con información mutua distinta de cero.

# 5. Scatter Plots de features pre-seleccionadas vs Target.

In [ ]:
get_scatter_plots(df_train, features=list_mi_feature_selected)

NOTAS:
1. Scatter plots de las features pre-selecionadas vs el Target se han graficado para ayudar a observar las verdaderas relaciones, lineales o no lineales.
2. En cada uno de estos plots se ha dibujado la línea de tendencia no paramétrica LOWESS para reveler relaciones complejas.
3. Visualmente, se confirma la fuerte relación no lineal con el Target como, por ejemplo, sucede en features tales como la 13 y la 18.
4. Para capturar estos comportamientos, se requerirá a modelos más allá de una regresión lineal.
5. No se observa la presencia de outliers unidimensionales ni el en Target ni en las features.

## 6. Distribución de features (Histograma + Boxplot) y Target.

In [ ]:
get_histograms(df_train, features=list_mi_feature_selected)

NOTAS:
1. Usando histograma+boxplot, se apoya visualmente a la no presencia de outliers unidimencionales en Target y features.

# 7. Comparación de distribuciones Train vs Test (Covariate Shift).


In [ ]:
get_train_test_comparison(df_train, df_test, features=list_mi_feature_selected)

NOTAS:
1. Se comparan las distribuciones de cada feature entre el set de training (azul) y el set test (rojo) para detectar posibles diferencias o drift.
2. Si las distribuciones son similares, un buen modelo debería generalizar bien el test.

# 8. Guardado de archivos.

In [ ]:
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

config["features"]["selected_by_mi"] = list_mi_feature_selected

with open("config.yaml", "w") as f:
    yaml.dump(config, f, allow_unicode=True, sort_keys=False)

print("Lista de features guardada en config.yaml:")
print(list_mi_feature_selected)


# 9. Conclusiones.

1. Los datos están en estado ideal para ser trabajados sin proprocesamiento previo. 
2. La estructura de dependencia  general con el target es no lineal y compleja. Por ende, considerar modelos  basados en árboles con boosting pueden ser buen comienzo para estos casos.